# Codex 多账号后台保活测试

这个 notebook 用来手动测试 `/Users/wangbin/.codex-*` 账号池的后台 tmux 保活。

它不会调用账号切换接口，也不会上传或覆盖任何 `auth.json`。会做的事包括：更新 LaunchAgent、立即运行保活脚本、查看 tmux 会话、查看日志，以及手动触发一次 `你好` 保活输入测试。

In [13]:
from pathlib import Path
import os
import platform
import shlex
import subprocess
import sys
import time
import urllib.request

APP_DIR = Path.cwd()
if not (APP_DIR / 'app.py').exists():
    APP_DIR = Path('/Users/wangbin/Documents/Codex/codex-account-switcher')
SCRIPT = APP_DIR / 'scripts' / 'ensure_codex_keepalive.sh'
WINDOWS_SCRIPT = APP_DIR / 'scripts' / 'ensure_codex_keepalive_windows.ps1'
PLIST = APP_DIR / 'launchd' / 'com.shizaishiwo.codex-keepalive.plist'
INSTALLED_PLIST = Path.home() / 'Library' / 'LaunchAgents' / 'com.shizaishiwo.codex-keepalive.plist'
LOG_FILE = APP_DIR / 'logs' / 'codex-keepalive.log'
WEB_LOG_FILE = APP_DIR / 'logs' / 'account-switcher-notebook.log'
APP_HOST = '127.0.0.1'
APP_PORT = 8765  # 可以在这里改网页端口，例如 9876
ACCOUNT_SEARCH_ROOT = Path(os.environ.get('CODEX_ACCOUNT_SEARCH_ROOT') or str(Path.home()))
LABEL = f'gui/{os.getuid()}/com.shizaishiwo.codex-keepalive' if hasattr(os, 'getuid') else 'com.shizaishiwo.codex-keepalive'


def run(cmd, *, check=False, env=None):
    print('$', cmd if isinstance(cmd, str) else ' '.join(shlex.quote(str(x)) for x in cmd))
    result = subprocess.run(cmd, shell=isinstance(cmd, str), text=True, capture_output=True, env=env, cwd=APP_DIR)
    if result.stdout:
        print(result.stdout.rstrip())
    if result.stderr:
        print(result.stderr.rstrip())
    if check and result.returncode != 0:
        raise RuntimeError(f'command failed with exit code {result.returncode}')
    print('exit:', result.returncode)
    return result


def app_url(path=''):
    suffix = path if path.startswith('/') or not path else '/' + path
    return f'http://{APP_HOST}:{APP_PORT}{suffix}'


def api_is_ready():
    try:
        with urllib.request.urlopen(app_url('/api/accounts'), timeout=3) as response:
            return response.status == 200
    except Exception:
        return False


def start_web_app():
    if api_is_ready():
        print('网页端已经可用:', app_url())
        return None
    env = os.environ.copy()
    env['CODEX_SWITCHER_HOST'] = APP_HOST
    env['CODEX_SWITCHER_PORT'] = str(APP_PORT)
    env.setdefault('CODEX_ACCOUNT_SEARCH_ROOT', str(ACCOUNT_SEARCH_ROOT))
    WEB_LOG_FILE.parent.mkdir(parents=True, exist_ok=True)
    log = WEB_LOG_FILE.open('a', encoding='utf-8')
    process = subprocess.Popen([sys.executable, str(APP_DIR / 'app.py')], cwd=APP_DIR, env=env, stdout=log, stderr=subprocess.STDOUT, text=True)
    for _ in range(30):
        if api_is_ready():
            print('网页端已启动:', app_url())
            print('PID:', process.pid)
            print('日志:', WEB_LOG_FILE)
            return process
        if process.poll() is not None:
            raise RuntimeError(f'网页端启动失败，退出码: {process.returncode}，请查看日志: {WEB_LOG_FILE}')
        time.sleep(1)
    raise RuntimeError(f'网页端 30 秒内没有响应，请查看日志: {WEB_LOG_FILE}')

print('APP_DIR =', APP_DIR)
print('SCRIPT =', SCRIPT)
print('WINDOWS_SCRIPT =', WINDOWS_SCRIPT)
print('PLIST =', PLIST)
print('INSTALLED_PLIST =', INSTALLED_PLIST)
print('LABEL =', LABEL)
print('ACCOUNT_SEARCH_ROOT =', ACCOUNT_SEARCH_ROOT)
print('WEB URL =', app_url())
print('OS =', platform.system())


APP_DIR = /Users/wangbin/Documents/Codex/codex-account-switcher
SCRIPT = /Users/wangbin/Documents/Codex/codex-account-switcher/scripts/ensure_codex_keepalive.sh
PLIST = /Users/wangbin/Documents/Codex/codex-account-switcher/launchd/com.shizaishiwo.codex-keepalive.plist
INSTALLED_PLIST = /Users/wangbin/Library/LaunchAgents/com.shizaishiwo.codex-keepalive.plist
LABEL = gui/501/com.shizaishiwo.codex-keepalive


## 1. 基础检查

确认脚本、plist、tmux、codex 都存在，并检查脚本语法和 plist 格式。

In [14]:
assert APP_DIR.exists(), APP_DIR
assert SCRIPT.exists(), SCRIPT
assert PLIST.exists(), PLIST

run(['bash', '-n', str(SCRIPT)], check=True)
run(['plutil', '-lint', str(PLIST)], check=True)
run(['which', 'tmux'])
run(['which', 'codex'])
run(['codex', '--version'])

$ bash -n /Users/wangbin/Documents/Codex/codex-account-switcher/scripts/ensure_codex_keepalive.sh
exit: 0
$ plutil -lint /Users/wangbin/Documents/Codex/codex-account-switcher/launchd/com.shizaishiwo.codex-keepalive.plist
/Users/wangbin/Documents/Codex/codex-account-switcher/launchd/com.shizaishiwo.codex-keepalive.plist: OK
exit: 0
$ which tmux
/opt/homebrew/bin/tmux
exit: 0
$ which codex
/opt/homebrew/bin/codex
exit: 0
$ codex --version
codex-cli 0.136.0
exit: 0


CompletedProcess(args=['codex', '--version'], returncode=0, stdout='codex-cli 0.136.0\n', stderr='')

## 1A. 启动本地网页端检测接口

这个 cell 会在本地启动网页端，并检测 `/api/accounts` 是否可访问。端口在第一个代码 cell 的 `APP_PORT` 里设置。


In [ ]:
web_process = start_web_app()
run([sys.executable, '-c', f"import urllib.request; print(urllib.request.urlopen('{app_url('/api/accounts')}', timeout=5).status)"], check=True)


## 2. 扫描账号池

只扫描 `/Users/wangbin/.codex-*`，不会包含默认 `/Users/wangbin/.codex`。

In [15]:
accounts = sorted(p for p in Path('/Users/wangbin').glob('.codex-*') if p.is_dir())
print('发现账号池目录数量:', len(accounts))
for p in accounts:
    auth = p / 'auth.json'
    config = p / 'config.toml'
    print(f'{p.name:32s} auth={auth.exists()} config={config.exists()}')

发现账号池目录数量: 10
.codex-shizaishiwo1123           auth=True config=True
.codex-shizaishiwo1223           auth=True config=True
.codex-shizaishiwo123            auth=True config=True
.codex-shizaishiwo1323           auth=True config=True
.codex-shizaishiwo2              auth=True config=True
.codex-shizaishiwo4              auth=True config=True
.codex-shizaishiwo5              auth=True config=True
.codex-shizaishiwo6              auth=True config=True
.codex-shizaishiwo7              auth=False config=False
.codex-zshi0509                  auth=True config=True


## 3. 更新并重新加载开机自启 LaunchAgent

这一步会把仓库里的最新 plist 复制到 `~/Library/LaunchAgents`，然后重新加载 `com.shizaishiwo.codex-keepalive`。

In [16]:
INSTALLED_PLIST.parent.mkdir(parents=True, exist_ok=True)
run(['cp', str(PLIST), str(INSTALLED_PLIST)], check=True)
run(['launchctl', 'bootout', f'gui/{os.getuid()}/com.shizaishiwo.codex-keepalive'])
run(['launchctl', 'bootstrap', f'gui/{os.getuid()}', str(INSTALLED_PLIST)], check=True)
run(['launchctl', 'print', LABEL])

$ cp /Users/wangbin/Documents/Codex/codex-account-switcher/launchd/com.shizaishiwo.codex-keepalive.plist /Users/wangbin/Library/LaunchAgents/com.shizaishiwo.codex-keepalive.plist
exit: 0
$ launchctl bootout gui/501/com.shizaishiwo.codex-keepalive
exit: 0
$ launchctl bootstrap gui/501 /Users/wangbin/Library/LaunchAgents/com.shizaishiwo.codex-keepalive.plist
exit: 0
$ launchctl print gui/501/com.shizaishiwo.codex-keepalive
gui/501/com.shizaishiwo.codex-keepalive = {
	active count = 1
	path = /Users/wangbin/Library/LaunchAgents/com.shizaishiwo.codex-keepalive.plist
	type = LaunchAgent
	state = xpcproxy

	program = /bin/bash
	arguments = {
		/bin/bash
		/Users/wangbin/Documents/Codex/codex-account-switcher/scripts/ensure_codex_keepalive.sh
	}

	working directory = /Users/wangbin/Documents/Codex/codex-account-switcher

	stdout path = /Users/wangbin/Documents/Codex/codex-account-switcher/logs/launchd.out.log
	stderr path = /Users/wangbin/Documents/Codex/codex-account-switcher/logs/launchd.er

CompletedProcess(args=['launchctl', 'print', 'gui/501/com.shizaishiwo.codex-keepalive'], returncode=0, stdout='gui/501/com.shizaishiwo.codex-keepalive = {\n\tactive count = 1\n\tpath = /Users/wangbin/Library/LaunchAgents/com.shizaishiwo.codex-keepalive.plist\n\ttype = LaunchAgent\n\tstate = xpcproxy\n\n\tprogram = /bin/bash\n\targuments = {\n\t\t/bin/bash\n\t\t/Users/wangbin/Documents/Codex/codex-account-switcher/scripts/ensure_codex_keepalive.sh\n\t}\n\n\tworking directory = /Users/wangbin/Documents/Codex/codex-account-switcher\n\n\tstdout path = /Users/wangbin/Documents/Codex/codex-account-switcher/logs/launchd.out.log\n\tstderr path = /Users/wangbin/Documents/Codex/codex-account-switcher/logs/launchd.err.log\n\tinherited environment = {\n\t\tSSH_AUTH_SOCK => /private/tmp/com.apple.launchd.8e2x9grMxU/Listeners\n\t}\n\n\tdefault environment = {\n\t\tPATH => /usr/bin:/bin:/usr/sbin:/sbin\n\t}\n\n\tenvironment = {\n\t\tOSLogRateLimit => 64\n\t\tXPC_SERVICE_NAME => com.shizaishiwo.codex-

## 4. 立即运行一次保活脚本

这一步就是快速启动所有符合条件账号的后台监控。脚本会：

- 自动扫描 `/Users/wangbin/.codex-*`
- 给每个账号写入 `/Users/wangbin` trusted 配置
- 对缺失或明显无效的 `auth.json` 写日志并跳过
- 为允许保活的账号启动 `tmux` 会话

In [17]:
run([str(SCRIPT)], check=True)

$ /Users/wangbin/Documents/Codex/codex-account-switcher/scripts/ensure_codex_keepalive.sh
[2026-06-03 10:01:17] ok codex-shizaishiwo1123: already running
[2026-06-03 10:01:17] ok codex-shizaishiwo1223: already running
[2026-06-03 10:01:17] ok codex-shizaishiwo123: already running
[2026-06-03 10:01:18] ok codex-shizaishiwo1323: already running
[2026-06-03 10:01:18] start codex-shizaishiwo2 with CODEX_HOME=/Users/wangbin/.codex-shizaishiwo2
[2026-06-03 10:01:18] ok codex-shizaishiwo4: already running
[2026-06-03 10:01:18] ok codex-shizaishiwo5: already running
[2026-06-03 10:01:18] ok codex-shizaishiwo6: already running
[2026-06-03 10:01:19] skip codex-shizaishiwo7: missing /Users/wangbin/.codex-shizaishiwo7/auth.json
[2026-06-03 10:01:19] ok codex-zshi0509: already running
[2026-06-03 10:01:19] done
exit: 0


CompletedProcess(args=['/Users/wangbin/Documents/Codex/codex-account-switcher/scripts/ensure_codex_keepalive.sh'], returncode=0, stdout='[2026-06-03 10:01:17] ok codex-shizaishiwo1123: already running\n[2026-06-03 10:01:17] ok codex-shizaishiwo1223: already running\n[2026-06-03 10:01:17] ok codex-shizaishiwo123: already running\n[2026-06-03 10:01:18] ok codex-shizaishiwo1323: already running\n[2026-06-03 10:01:18] start codex-shizaishiwo2 with CODEX_HOME=/Users/wangbin/.codex-shizaishiwo2\n[2026-06-03 10:01:18] ok codex-shizaishiwo4: already running\n[2026-06-03 10:01:18] ok codex-shizaishiwo5: already running\n[2026-06-03 10:01:18] ok codex-shizaishiwo6: already running\n[2026-06-03 10:01:19] skip codex-shizaishiwo7: missing /Users/wangbin/.codex-shizaishiwo7/auth.json\n[2026-06-03 10:01:19] ok codex-zshi0509: already running\n[2026-06-03 10:01:19] done\n', stderr='')

## 5. 查看 tmux 会话

确认是否出现 `codex-账号id` 形式的 session。

In [18]:
run(['tmux', 'list-sessions', '-F', '#{session_name} #{session_created_string}'])

result = subprocess.run(['tmux', 'list-sessions', '-F', '#{session_name}'], text=True, capture_output=True)
sessions = [line.strip() for line in result.stdout.splitlines() if line.strip().startswith('codex-')]
print('\nCodex keepalive sessions:', len(sessions))
for s in sessions:
    print('-', s)

$ tmux list-sessions -F '#{session_name} #{session_created_string}'
codex-shizaishiwo1123 
codex-shizaishiwo1223 
codex-shizaishiwo123 
codex-shizaishiwo1323 
codex-shizaishiwo2 
codex-shizaishiwo4 
codex-shizaishiwo5 
codex-shizaishiwo6 
codex-zshi0509
exit: 0

Codex keepalive sessions: 9
- codex-shizaishiwo1123
- codex-shizaishiwo1223
- codex-shizaishiwo123
- codex-shizaishiwo1323
- codex-shizaishiwo2
- codex-shizaishiwo4
- codex-shizaishiwo5
- codex-shizaishiwo6
- codex-zshi0509


## 6. 查看保活脚本日志

重点看是否有 `start codex-*`、`already running`、`invalid or expired-looking auth.json`。

In [19]:
if LOG_FILE.exists():
    lines = LOG_FILE.read_text(encoding='utf-8', errors='replace').splitlines()
    for line in lines[-120:]:
        print(line)
else:
    print('日志文件还不存在:', LOG_FILE)

[2026-06-03 09:48:43] done
[2026-06-03 09:48:43] skip codex-bad: invalid or expired-looking auth.json
[2026-06-03 09:48:43] done
[2026-06-03 09:48:44] start codex-alpha with CODEX_HOME=/var/folders/06/xc77v9q12hz80mbtrkq97_b40000gn/T/tmpwutx9rr8/.codex-alpha
[2026-06-03 09:48:44] start codex-beta with CODEX_HOME=/var/folders/06/xc77v9q12hz80mbtrkq97_b40000gn/T/tmpwutx9rr8/.codex-beta
[2026-06-03 09:48:44] done
[2026-06-03 09:49:32] ok codex-alpha: already running
[2026-06-03 09:49:32] daily ping codex-alpha
[2026-06-03 09:49:32] done
[2026-06-03 09:49:32] ok codex-alpha: already running
[2026-06-03 09:49:32] done
[2026-06-03 09:49:32] skip codex-bad: invalid or expired-looking auth.json
[2026-06-03 09:49:32] done
[2026-06-03 09:49:32] start codex-alpha with CODEX_HOME=/var/folders/06/xc77v9q12hz80mbtrkq97_b40000gn/T/tmppxr38806/.codex-alpha
[2026-06-03 09:49:32] start codex-beta with CODEX_HOME=/var/folders/06/xc77v9q12hz80mbtrkq97_b40000gn/T/tmppxr38806/.codex-beta
[2026-06-03 09:49:3

## 7. 检查每个账号的 trusted 配置

确认每个账号的 `config.toml` 里都有 `[projects."/Users/wangbin"] trust_level = "trusted"`。

In [20]:
expected_header = '[projects."/Users/wangbin"]'
for p in accounts:
    config = p / 'config.toml'
    if not config.exists():
        print(p.name, '缺少 config.toml')
        continue
    text = config.read_text(encoding='utf-8', errors='replace')
    ok = expected_header in text and 'trust_level = "trusted"' in text
    print(f'{p.name:32s}', 'OK' if ok else '需要检查')

.codex-shizaishiwo1123           OK
.codex-shizaishiwo1223           OK
.codex-shizaishiwo123            OK
.codex-shizaishiwo1323           OK
.codex-shizaishiwo2              OK
.codex-shizaishiwo4              OK
.codex-shizaishiwo5              OK
.codex-shizaishiwo6              OK
.codex-shizaishiwo7 缺少 config.toml
.codex-zshi0509                  OK


## 8. 手动触发一次“你好”保活输入测试

正常逻辑只在每天本地时间 00 点这一小时发送一次。这个 cell 用临时状态文件和当前小时强制测试一次，不会影响正式的 `keepalive_ping_state.json`。

In [21]:
env = os.environ.copy()
env['KEEPALIVE_DAILY_PING_HOUR'] = time.strftime('%H')
env['KEEPALIVE_PING_STATE_FILE'] = f'/tmp/codex-keepalive-ping-test-{int(time.time())}.json'
run([str(SCRIPT)], check=True, env=env)
print('临时 ping state:', env['KEEPALIVE_PING_STATE_FILE'])

$ /Users/wangbin/Documents/Codex/codex-account-switcher/scripts/ensure_codex_keepalive.sh
[2026-06-03 10:01:19] ok codex-shizaishiwo1123: already running
[2026-06-03 10:01:19] daily ping codex-shizaishiwo1123
[2026-06-03 10:01:19] ok codex-shizaishiwo1223: already running
[2026-06-03 10:01:19] daily ping codex-shizaishiwo1223
[2026-06-03 10:01:19] ok codex-shizaishiwo123: already running
[2026-06-03 10:01:19] daily ping codex-shizaishiwo123
[2026-06-03 10:01:20] ok codex-shizaishiwo1323: already running
[2026-06-03 10:01:20] daily ping codex-shizaishiwo1323
[2026-06-03 10:01:20] ok codex-shizaishiwo2: already running
[2026-06-03 10:01:20] daily ping codex-shizaishiwo2
[2026-06-03 10:01:20] ok codex-shizaishiwo4: already running
[2026-06-03 10:01:20] daily ping codex-shizaishiwo4
[2026-06-03 10:01:20] ok codex-shizaishiwo5: already running
[2026-06-03 10:01:20] daily ping codex-shizaishiwo5
[2026-06-03 10:01:21] ok codex-shizaishiwo6: already running
[2026-06-03 10:01:21] daily ping cod

RuntimeError: command failed with exit code 2

## 9. 抽样查看 tmux pane 输出

用于确认 Codex CLI 是否卡在提示、登录失败、或已经进入交互界面。这里只截取每个 `codex-*` session 最近 80 行。

In [ ]:
result = subprocess.run(['tmux', 'list-sessions', '-F', '#{session_name}'], text=True, capture_output=True)
sessions = [line.strip() for line in result.stdout.splitlines() if line.strip().startswith('codex-')]
for s in sessions:
    print('\n' + '=' * 20, s, '=' * 20)
    run(['tmux', 'capture-pane', '-p', '-t', s, '-S', '-200'])


==================== codex-shizaishiwo1123 ====================
$ tmux capture-pane -p -t codex-shizaishiwo1123 -S -200

⚠ Ignored unsupported project-local config keys in /Users/wangbin/.codex/
  config.toml: notify. If you want these settings to apply, manually set them in
  your user-level config.toml.

╭────────────────────────────────────────────────╮
│ >_ OpenAI Codex (v0.135.0)                     │
│                                                │
│ model:       gpt-5.5 medium   /model to change │
│ directory:   ~                                 │
│ permissions: YOLO mode                         │
╰────────────────────────────────────────────────╯

  Tip: GPT-5.5 is now available in Codex. It's our strongest agentic coding
  model yet, built to reason through large codebases, check assumptions with
  tools, and keep going until the work is done.

  Learn more: https://openai.com/index/introducing-gpt-5-5/

⚠ Ignored unsupported project-local config keys in /Users/wangbin/.cod

## 10. 再次查看 LaunchAgent 实际生效配置

确认路径已经是当前项目目录，不再是旧的 `任意任务` 路径。

In [ ]:
run(['launchctl', 'print', LABEL])

$ launchctl print gui/501/com.shizaishiwo.codex-keepalive
gui/501/com.shizaishiwo.codex-keepalive = {
	active count = 0
	path = /Users/wangbin/Library/LaunchAgents/com.shizaishiwo.codex-keepalive.plist
	type = LaunchAgent
	state = not running

	program = /bin/bash
	arguments = {
		/bin/bash
		/Users/wangbin/Documents/Codex/codex-account-switcher/scripts/ensure_codex_keepalive.sh
	}

	working directory = /Users/wangbin/Documents/Codex/codex-account-switcher

	stdout path = /Users/wangbin/Documents/Codex/codex-account-switcher/logs/launchd.out.log
	stderr path = /Users/wangbin/Documents/Codex/codex-account-switcher/logs/launchd.err.log
	inherited environment = {
		SSH_AUTH_SOCK => /private/tmp/com.apple.launchd.8e2x9grMxU/Listeners
	}

	default environment = {
		PATH => /usr/bin:/bin:/usr/sbin:/sbin
	}

	environment = {
		OSLogRateLimit => 64
		XPC_SERVICE_NAME => com.shizaishiwo.codex-keepalive
	}

	domain = gui/501 [100022]
	asid = 100022
	minimum runtime = 10
	exit timeout = 5
	runs =

CompletedProcess(args=['launchctl', 'print', 'gui/501/com.shizaishiwo.codex-keepalive'], returncode=0, stdout='gui/501/com.shizaishiwo.codex-keepalive = {\n\tactive count = 0\n\tpath = /Users/wangbin/Library/LaunchAgents/com.shizaishiwo.codex-keepalive.plist\n\ttype = LaunchAgent\n\tstate = not running\n\n\tprogram = /bin/bash\n\targuments = {\n\t\t/bin/bash\n\t\t/Users/wangbin/Documents/Codex/codex-account-switcher/scripts/ensure_codex_keepalive.sh\n\t}\n\n\tworking directory = /Users/wangbin/Documents/Codex/codex-account-switcher\n\n\tstdout path = /Users/wangbin/Documents/Codex/codex-account-switcher/logs/launchd.out.log\n\tstderr path = /Users/wangbin/Documents/Codex/codex-account-switcher/logs/launchd.err.log\n\tinherited environment = {\n\t\tSSH_AUTH_SOCK => /private/tmp/com.apple.launchd.8e2x9grMxU/Listeners\n\t}\n\n\tdefault environment = {\n\t\tPATH => /usr/bin:/bin:/usr/sbin:/sbin\n\t}\n\n\tenvironment = {\n\t\tOSLogRateLimit => 64\n\t\tXPC_SERVICE_NAME => com.shizaishiwo.cod